## 상품 CRUD - products

In [ ]:
import os
from supabase import create_client
from dotenv import load_dotenv
load_dotenv()

RUN_PRODUCT_DEMO = False  # 회원가입/로그인 단독 실습을 할 때만 True로 변경

In [ ]:
if RUN_PRODUCT_DEMO:
    url = os.getenv("SUPABASE_URL")
    key = os.getenv("SUPABASE_PUBLISHABLE_KEY")
    supabase = create_client(url, key)

In [ ]:
if RUN_PRODUCT_DEMO:
    email = input("가입할 이메일: ")
    password = input("가입할 비밀번호: ")
    sign_in = supabase.auth.sign_up({"email": email, "password": password})

In [ ]:
if RUN_PRODUCT_DEMO:
    log_in = supabase.auth.sign_in_with_password({"email": email, "password": password})
    password = None
    print("로그인한 회원 ID:", log_in.user.id)

In [ ]:
if RUN_PRODUCT_DEMO:
    login_user = supabase.auth.get_user()
    print("로그인 확인:", login_user.user.id)

In [ ]:
class product_service: # create, update, delete 등등

    def __init__ (self, supabase) :
        self.supabase = supabase
        user_info = self.supabase.auth.get_user()

        if not user_info or not user_info.user :
            raise PermissionError("로그인 후 다시 시도하세요")

        self.user = user_info.user


    #~~[create] - supabase table editor로 products 테이블 구조 만듦~~


    # [insert] - 신규 상품등록
    def new_product(self, name, price, description) :

        # 1. 현재 로그인한 사용자의 회원 유형 조회
        user_detail = (
            self.supabase.table("user_details")
                .select("type")
                .eq("id", self.user.id)
                .is_("deleted_at", "null")
                .execute()
        )

        # 2. 회원 유형이 seller인 경우만 상품 등록할 수 있도록
        if not user_detail.data :
            raise PermissionError("회원가입을 먼저 진행해주세요")

        if user_detail.data[0]["type"] != "SELLER" :
            raise PermissionError("Seller로 등록된 회원이 아닙니다")

        # 3. seller 회원 - 상품등록
        new_product = (
            self.supabase.table('products')
                .insert({
                    "name" : name,
                    "price" : price,
                    "description" : description,
                    "seller_id" : self.user.id
                })
                .execute()
        )
        return new_product.data


    # [select] - 상품 조회
    def get_product(self, name) :
        get_product = (
            self.supabase.table("products")
                .select("id,name,price,description,seller_id")
                .ilike("name", f"%{name}%")
                .is_("deleted_at", "null")
                .execute()
        )

        return get_product.data


    # [update] - 상품명 수정
    def modify_product_name(self, id, name) :
        responce_product_name = (
            self.supabase.table("products")
            .update({
                "name" : name
                })
                .eq("id", id)
                .eq("seller_id", self.user.id) # seller만 수정 가능하도록
                .execute()
        )

        return f"수정된 [[상품명]] 데이터 : {responce_product_name.data}"


    # [update] - 상품가격 수정
    def modify_product_price(self, id, price) :
        response_product_price = (
            self.supabase.table("products")
            .update({
                "price" : price
            })
            .eq("id", id)
            .eq("seller_id", self.user.id) # seller만 수정 가능하도록
            .execute()
        )

        return f"수정된 [[가격]] 데이터 : {response_product_price.data}"


     # [update] - 상품 설명 수정
    def modify_product_description(self, id, description) :
        response_product_description = (
            self.supabase.table("products")
            .update({
                 "description" : description
            })
            .eq("id", id)
            .eq("seller_id", self.user.id) # seller만 수정 가능하도록
            .execute()
        )

        return f"수정된 [[상품설명]] 데이터 : {response_product_description.data}"